# Semantic Chunking: Break Where Topics Change

| Field | Value |
|---|---|
| Stage | Chunking |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Semantic chunking is a breakpoint policy, not magic. Compare it with a simple baseline on answer-support coverage and context noise.

## 30-Second Summary

This notebook splits a five-sentence fixture by paragraph baseline and by adjacent-sentence TF-IDF similarity. The semantic policy detects the topic change from LangChain to Paris while preserving every sentence exactly once.

## Why This Matters

Fixed boundaries can join unrelated topics or cut a coherent explanation. Semantic breakpoints can help, but thresholds add model, latency, and stability trade-offs.

## Scope

| Covers | Does not cover |
|---|---|
| Sentence boundaries, adjacent similarity, threshold selection, coverage checks | Neural semantic models, multilingual segmentation, long-document benchmark |


## Mental Model

```text
sentences -> adjacent similarity -> low score? break : merge -> chunks -> retrieval evaluation
```


In [1]:
from pathlib import Path
import re
from rag_101 import TfidfVectorizer, find_repo_root

REPO_ROOT = find_repo_root()
TEXT_PATH = REPO_ROOT / "08-advanced-chunking-and-preprocessing/langchain_intro.txt"
text = TEXT_PATH.read_text(encoding="utf-8").strip()
sentences = [item.strip() for item in re.split(r"(?<=[.!?])\s+", text) if item.strip()]
sentences


['LangChain is a framework for building applications with LLMs.',
 'Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.',
 'You can create chains, agents, memory, and retrievers.',
 'The Eiffel Tower is located in Paris.',
 'France is a popular tourist destination.']

## How It Works

We fit one vectorizer across sentences, compute adjacent cosine similarity, and start a new chunk when the score falls below a visible threshold. The threshold is a parameter to evaluate, not a universal constant.


## Baseline

The baseline groups two consecutive sentences per chunk. It is deterministic but ignores topic changes.


In [2]:
baseline_chunks = [" ".join(sentences[index:index + 2]) for index in range(0, len(sentences), 2)]
baseline_chunks


['LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.',
 'You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris.',
 'France is a popular tourist destination.']

## Technique Implementation

TF-IDF is lexical rather than truly semantic, but it makes breakpoint mechanics inspectable and offline. A production semantic model can replace the vectors while the policy and checks stay the same.


In [3]:
vectorizer = TfidfVectorizer().fit(sentences)
vectors = vectorizer.transform(sentences)
def dot(left, right): return sum(a * b for a, b in zip(left, right, strict=True))
adjacent_scores = [dot(vectors[index], vectors[index + 1]) for index in range(len(vectors) - 1)]
threshold = 0.05
semantic_chunks = [[sentences[0]]]
for index, sentence in enumerate(sentences[1:]):
    if adjacent_scores[index] < threshold:
        semantic_chunks.append([sentence])
    else:
        semantic_chunks[-1].append(sentence)
semantic_chunks = [" ".join(chunk) for chunk in semantic_chunks]
list(zip(range(1, len(adjacent_scores) + 1), adjacent_scores)), semantic_chunks


([(1, 0.20579935795320814), (2, 0.0), (3, 0.0), (4, 0.0)],
 ['LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.',
  'You can create chains, agents, memory, and retrievers.',
  'The Eiffel Tower is located in Paris.',
  'France is a popular tourist destination.'])

## Controlled Experiment

We compare whether chunks mix the two known topics (`LangChain` and `Paris/France`) and verify lossless sentence coverage. This fixture is intentionally tiny so every boundary is auditable.


In [4]:
def mixed_topic_count(chunks: list[str]) -> int:
    source_terms = ("langchain", "llm", "chain", "agent", "retriever")
    destination_terms = ("paris", "france", "eiffel")
    return sum(
        any(term in chunk.lower() for term in source_terms)
        and any(term in chunk.lower() for term in destination_terms)
        for chunk in chunks
    )

experiment_result = {
    "baseline_chunks": len(baseline_chunks),
    "semantic_chunks": len(semantic_chunks),
    "baseline_mixed_topics": mixed_topic_count(baseline_chunks),
    "semantic_mixed_topics": mixed_topic_count(semantic_chunks),
    "break_before_paris": adjacent_scores[2] < threshold,
}
experiment_result


{'baseline_chunks': 3,
 'semantic_chunks': 4,
 'baseline_mixed_topics': 1,
 'semantic_mixed_topics': 0,
 'break_before_paris': True}

## Evaluation

The fixed two-sentence baseline mixes topics once because sentence 4 (Paris) is paired with sentence 3 (retrievers). The similarity policy breaks before Paris and produces no mixed-topic chunk, but it also over-segments the LangChain discussion because adjacent sentences 2 and 3 share no terms after preprocessing. This is a useful failure case: lexical breakpoints are not semantic understanding.


In [5]:
assert [sentence for chunk in semantic_chunks for sentence in re.split(r"(?<=[.!?])\s+", chunk)] == sentences
assert experiment_result["baseline_mixed_topics"] == 1
assert experiment_result["semantic_mixed_topics"] == 0
assert experiment_result["break_before_paris"]
print("Semantic chunking checks passed.")


Semantic chunking checks passed.


## Decision Guide

| Situation | Strategy |
|---|---|
| Stable headings/sections | Structure-aware first |
| Short uniform prose | Token/recursive baseline |
| Topic shifts inside long prose | Evaluated semantic breakpoints |
| Need parent context | Small-to-big/parent retrieval |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Too many tiny chunks | Threshold too high | Plot score distribution and tune on labels |
| Topics still mix | Weak representation | Better model or structure signals |
| Updates rewrite all IDs | Boundary instability | Version chunks and use durable anchors |
| Retrieval improves but answers worsen | Context lost | Measure support coverage, not chunk purity alone |


## Production Notes

### Observability
Track chunk counts, size distribution, breakpoint scores, model/version, and retrieval metrics.

### Safety and Guardrails
Do not send sensitive documents to an unapproved embedding service. Preserve source permissions on every chunk.

### Latency and Cost
Semantic policies embed sentences and may be much costlier than structural or token splitting; cache by content hash.


## Practice

Add a bridging sentence that mentions both LangChain and Paris. Predict how each strategy changes and whether a mixed chunk is actually wrong.

## Recall

Toggle - Recall: What does the threshold control?
The trade-off between merging coherent neighbors and creating smaller chunks.

Toggle - Recall: What must semantic chunking preserve?
Complete source coverage, order, provenance, and answer-support context.

## Sources

- [LangChain text splitters](https://python.langchain.com/docs/concepts/text_splitters/)
- Repository fixture: `langchain_intro.txt`

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the transparent fixture experiment | Compare neural breakpoints on the shared golden corpus |
